# 02 — Image Processor

Generates only missing images. It reports every scene and resumes safely after a Colab reset.

In [ ]:
import os, sys, subprocess
ROOT="/content/black-history-factory"
REPO_URL="https://github.com/jonbBla/black-history-factory.git"
if not os.path.exists(ROOT):
    subprocess.run(["git","clone",REPO_URL,ROOT], check=True)
sys.path.insert(0, ROOT)
subprocess.run([sys.executable,"-m","pip","install","-q","diffusers","transformers","accelerate","safetensors"],check=True)
from factory.drive import mount_drive,DrivePaths
from factory.config import Config
MYDRIVE=mount_drive(); paths=DrivePaths(os.path.join(MYDRIVE,"BLACK_HISTORY_FACTORY")); paths.ensure_tree(); config=Config.load(paths.root)
print(f"[SETUP] Drive: {paths.root}")


In [ ]:
from factory.image_engine import load_sdxl_lightning
pipe=load_sdxl_lightning(config.image_model)
print("[IMAGE] SDXL-Lightning loaded.")


In [ ]:
from factory.image_engine import run
from factory.utils import read_json,write_json_atomic
from factory import status
def process_one():
    jobs=[]
    for jid in sorted(os.listdir(paths("02_JOBS"))):
        m=read_json(paths.manifest(jid),{}) or {}
        if m.get("status") in ("QWEN_READY","IMAGES_PARTIAL"): jobs.append(jid)
    if not jobs:
        print("[IMAGE] No QWEN_READY job."); return False
    jid=jobs[0]; scenes=read_json(paths.scenes(jid),[])
    status.set_processor(paths,"image","running",jid,"image_generation",f"0/{len(scenes)}",0,len(scenes))
    def progress(n,total): status.set_processor(paths,"image","running",jid,"image_generation",f"scene {n}/{total}",n,total)
    try:
        run(paths,jid,scenes,pipe,config,progress)
        m=read_json(paths.manifest(jid),{}) or {}; m["status"]="IMAGES_READY"; write_json_atomic(paths.manifest(jid),m)
        status.set_processor(paths,"image","idle",jid,"ready","images complete",len(scenes),len(scenes)); print(f"[IMAGE] COMPLETE {jid}"); return True
    except Exception as e:
        m=read_json(paths.manifest(jid),{}) or {}; m.update(status="IMAGES_PARTIAL",image_error=str(e)); write_json_atomic(paths.manifest(jid),m)
        status.set_processor(paths,"image","error",jid,"failed",str(e)); print(f"[IMAGE] ERROR {jid} | {e}"); return False
process_one()


In [ ]:
# Optional after the one-video test.
while process_one(): pass
